# Xử lý Dataset cho Fake News Detection

Notebook này sẽ xử lý dataset để chuẩn bị cho bài toán phát hiện tin giả (fake news detection) với học bán giám sát (semi-supervised learning).

Các bước thực hiện:
1. Import thư viện cần thiết
2. Load và làm sạch dataset
3. Gộp các đặc trưng thành cột text
4. Chia tập train/val/test
5. Tạo tập huấn luyện có nhãn và không nhãn
6. Lưu các tập dữ liệu đã xử lý

# 1. Import Thư viện

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import os
import re

# 2. Load và làm sạch dataset

In [2]:
# Đọc dataset
df = pd.read_csv('public_train.csv')

# Kiểm tra thông tin cơ bản
print("Thông tin dataset:")
print(df.info())
print("\nSố lượng mẫu cho mỗi nhãn:")
print(df['label'].value_counts())

Thông tin dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4372 entries, 0 to 4371
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                4372 non-null   int64 
 1   user_name         4372 non-null   object
 2   post_message      4371 non-null   object
 3   timestamp_post    4276 non-null   object
 4   num_like_post     4257 non-null   object
 5   num_comment_post  4362 non-null   object
 6   num_share_post    3647 non-null   object
 7   label             4372 non-null   int64 
dtypes: int64(2), object(6)
memory usage: 273.4+ KB
None

Số lượng mẫu cho mỗi nhãn:
label
0    3638
1     734
Name: count, dtype: int64


# 3. Gộp các đặc trưng thành cột text

In [3]:
# Chỉ giữ lại cột post_message làm text vì đây là nội dung chính của bài đăng
df['text'] = df['post_message']

# Chỉ giữ lại 2 cột cần thiết
df_processed = df[['text', 'label']]

print("Số dòng sau khi xử lý:", len(df_processed))
print("\nXem 5 mẫu đầu tiên:")
print(df_processed.head())

Số dòng sau khi xử lý: 4372

Xem 5 mẫu đầu tiên:
                                                text  label
0  THĂNG CẤP BẬC HÀM ĐỐI VỚI 2 CÁN BỘ, CHIẾN SỸ H...      0
1                                              <URL>      0
2  TƯ VẤN MÙA THI: Cách nộp hồ sơ để trúng tuyển ...      0
3  Cơ quan Cạnh tranh và Thị trường Anh quyết địn...      0
4  Thêm 7 ca tại Quảng Nam liên quan đến hành khá...      0


# 4. Chia tập train/val/test

In [4]:
# Đầu tiên chia thành train và temp (temp sẽ được chia thành val và test)
train_df, temp_df = train_test_split(df_processed, test_size=0.4, stratify=df_processed['label'], random_state=42)

# Chia temp thành val và test với tỷ lệ bằng nhau
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print("Kích thước các tập:")
print(f"Train: {len(train_df)} mẫu")
print(f"Val: {len(val_df)} mẫu")
print(f"Test: {len(test_df)} mẫu")

Kích thước các tập:
Train: 2623 mẫu
Val: 874 mẫu
Test: 875 mẫu


# 5. Tạo tập huấn luyện có nhãn và không nhãn

In [5]:
# Lấy 10 mẫu đầu tiên làm tập có nhãn
labeled_train_df = train_df.iloc[:10].copy()

# Phần còn lại làm tập không nhãn
unlabeled_train_df = train_df.iloc[10:].copy()

# Tạo backup của tập không nhãn trước khi xóa nhãn
unlabeled_backup_df = unlabeled_train_df.copy()

# Xóa nhãn của tập unlabeled
unlabeled_train_df['label'] = -1

print("Kích thước các tập:")
print(f"Tập train có nhãn: {len(labeled_train_df)} mẫu")
print(f"Tập train không nhãn: {len(unlabeled_train_df)} mẫu")
print("\nPhân bố nhãn trong tập có nhãn:")
print(labeled_train_df['label'].value_counts())

Kích thước các tập:
Tập train có nhãn: 10 mẫu
Tập train không nhãn: 2613 mẫu

Phân bố nhãn trong tập có nhãn:
label
0    7
1    3
Name: count, dtype: int64


# 6. Lưu các tập dữ liệu đã xử lý

In [6]:
# Tạo thư mục để lưu nếu chưa tồn tại
os.makedirs('processed', exist_ok=True)

# Lưu các tập dữ liệu
val_df.to_csv('processed/val.csv', index=False)
test_df.to_csv('processed/test.csv', index=False)
labeled_train_df.to_csv('processed/train_labeled.csv', index=False)
unlabeled_train_df.to_csv('processed/train_unlabeled.csv', index=False)
unlabeled_backup_df.to_csv('processed/train_unlabeled_backup.csv', index=False)

print("Đã lưu các tập dữ liệu vào thư mục 'processed':")
print("1. val.csv")
print("2. test.csv")
print("3. train_labeled.csv")
print("4. train_unlabeled.csv")
print("5. train_unlabeled_backup.csv")

Đã lưu các tập dữ liệu vào thư mục 'processed':
1. val.csv
2. test.csv
3. train_labeled.csv
4. train_unlabeled.csv
5. train_unlabeled_backup.csv
